# 数据准备

读取原始数据、统计分布与长度、清洗描述，并保存处理数据及划分。


## 1. 导入与路径配置


In [ ]:
import csv
import hashlib
import json
import math
import random
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

PROJECT = Path(r"D:\Pycharm\SemEval_27")
DATA_DIR = PROJECT / "data"


## 2. 数据处理函数


### 2.1 标签映射与文件保存


In [ ]:
IDENTITIES = ['l', 'g', 'b', 't', 'q', 'i', 'a', 'nb', 'lgbtqia+']
HATE_MAP = {'no': 0, 'yes_implicit': 1, 'yes_explicit': 2}
STEREOTYPE = {'no': 0, 'yes': 1}
SCOPE = {'group': 0, 'individual': 1}

In [ ]:
def write_data_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

### 2.2 描述清洗与内容统计


In [ ]:
PROMO_PATTERNS = {
    'subscribe_en': r'^(?:please\s+)?subscribe(?:\s+(?:here|now|to\s+(?:our|the)\s+channel))?[\s:!→-]*(?:\[URL\]|https?://\S+)?[\s.!]*$',
    'subscribe_it': r'^iscriviti\s+al\s+(?:nostro\s+)?canale[\s:!→-]*(?:\[URL\]|https?://\S+)?[\s.!]*$',
    'subscribe_nl': r'^(?:meer\s+van\s+onze\s+video.s\s+zien\?\s*)?abonneer\s+(?:je\s+hier|op\s+ons\s+kanaal)[\s:!→-]*(?:\[URL\]|https?://\S+)?[\s.!]*$',
    'social_link': r'^(?:facebook|twitter|instagram|tiktok|site|app)\s*:\s*(?:\[URL\]|https?://\S+|www\.\S+)\s*$',
    'website_link': r'^(?:check\s+out\s+our\s+website|collegati\s+alla\s+home\s+page)\s*:\s*(?:\[URL\]|https?://\S+)\s*$',
}

def clean_description_block(records, enabled=False):
    # 清理完整推广行；返回副本，保留原文及上下文组编号。
    cleaned_records, counts = [], defaultdict(Counter)
    rules = {name: re.compile(pattern, re.I) for name, pattern in PROMO_PATTERNS.items()}
    for original_row in records:
        row = dict(original_row)
        removed, kept = [], []
        if enabled:
            for line in row['description'].splitlines(keepends=True):
                candidate = re.sub(r'^\s*[-•]\s*', '', line.strip())
                reason = next((name for name, rule in rules.items() if rule.fullmatch(candidate)), None)
                if reason:
                    removed.append(line)
                else:
                    kept.append(line)
            description = ''.join(kept)
            if removed and description.strip():
                row.setdefault('raw_description', row['description'])
                row['description'] = description
                counts[row['language']]['changed_rows'] += 1
                counts[row['language']]['removed_lines'] += len(removed)
                counts[row['language']]['removed_characters'] += len(original_row['description'])-len(description)
            elif removed:
                counts[row['language']]['empty_result_kept_original'] += 1
        cleaned_records.append(row)
    return cleaned_records, {language: dict(values) for language, values in counts.items()}


def audit_content(records):
    contexts = {r['group_id']: r for r in records}
    patterns = Counter()
    rules = {name: re.compile(pattern, re.I) for name, pattern in PROMO_PATTERNS.items()}
    for row in contexts.values():
        for line in row['description'].splitlines():
            candidate = re.sub(r'^\s*[-•]\s*', '', line.strip())
            for name, rule in rules.items():
                if rule.fullmatch(candidate): patterns[name] += 1
    duplicates = {}
    normalizers = {'exact': lambda s:s, 'whitespace': lambda s:' '.join(s.split()),
                   'casefold': lambda s:' '.join(s.split()).casefold()}
    for name, normalizer in normalizers.items():
        groups = defaultdict(list)
        for row in records: groups[(row['language'], normalizer(row['comment']))].append(row['id'])
        duplicates[name] = {'extra_rows':sum(len(ids)-1 for ids in groups.values()),
                            'groups':sum(len(ids)>1 for ids in groups.values())}
    return {'context_count':len(contexts), 'promotion_patterns':dict(patterns), 'duplicates':duplicates}


### 2.3 数据读取与标签编码


In [ ]:
def load_data(source):
    records = []
    for language, split in [('EN', 'training'), ('IT', 'training'), ('NL', 'training'), ('FA', 'development')]:
        folder = 'StereoQueerEval_training' if split == 'training' else 'StereoQueerEval_FA_development'
        path = source / folder / f'StereoQueerEval_{language}_{split}.tsv'
        with path.open(encoding='utf-8-sig', newline='') as handle:
            reader = csv.DictReader(handle, delimiter='\t', strict=True)
            rows = list(reader)
        for row in rows:
            sample_id = row['StereoQueerEval_id']
            hateful = row['hate_speech'] != 'no'
            scope_id, identity_labels = (-100, [0] * len(IDENTITIES))
            if hateful:
                scope, targets = row['target'].split('_', 1)
                names = targets.split(',')
                scope_id = SCOPE[scope]
                identity_labels = [int(name in names) for name in IDENTITIES]
            context = json.dumps([language, row['yt_title'], row['yt_description']], ensure_ascii=False)
            group_id = language + '_' + hashlib.sha256(context.encode('utf-8')).hexdigest()
            records.append({'id': sample_id, 'language': language, 'source_split': split, 'group_id': group_id, 'title': row['yt_title'], 'description': row['yt_description'], 'comment': row['yt_comment'], 'stereotype': row['stereotype'], 'hate_speech': row['hate_speech'], 'target': row['target'], 'stereotype_label': STEREOTYPE[row['stereotype']], 'hate_label': HATE_MAP[row['hate_speech']], 'scope_label': scope_id, 'identity_labels': identity_labels, 'target_loss_mask': int(hateful)})
    return records


### 2.4 分布统计


In [ ]:
def summarize(records):
    result = {'n': len(records), 'groups': len({r['group_id'] for r in records})}
    for field in ['language', 'stereotype', 'hate_speech', 'target']:
        result[field] = dict(sorted(Counter(r[field] for r in records).items()))
    result['identities'] = {name: sum(r['identity_labels'][i] for r in records)
                            for i, name in enumerate(IDENTITIES)}
    result['characters'] = {}
    for field in ['comment', 'title', 'description']:
        lengths = sorted(len(r[field]) for r in records)
        result['characters'][field] = {'median': statistics.median(lengths),
                                       'p95': lengths[int((len(lengths)-1)*.95)], 'max': max(lengths)}
    return result


### 2.5 分组划分与结果保存


In [ ]:
def grouped_holdout(rows, fraction, seed, attempts):
    """搜索可复现的分组划分，平衡样本数量及刻板印象、仇恨和联合标签分布。

    不使用文本语义、验证预测、对象标签或模型分数。
    这是近似分层，不保证类别比例完全一致。
    """
    groups = defaultdict(list)
    for r in rows:
        groups[r['group_id']].append(r)
    keys = sorted(groups)
    n_groups = max(1, min(len(keys)-1, round(len(keys)*fraction)))
    group_counts = {}
    totals = Counter()
    for key, members in groups.items():
        counts = Counter()
        for r in members:
            counts['size'] += 1
            counts['a:' + r['stereotype']] += 1
            counts['b:' + r['hate_speech']] += 1
            counts['ab:' + r['stereotype'] + ':' + r['hate_speech']] += 1
        group_counts[key] = counts
        totals.update(counts)
    rng = random.Random(seed)
    best, best_score = None, float('inf')
    for _ in range(attempts):
        chosen = set(rng.sample(keys, n_groups))
        counts = Counter()
        for key in sorted(chosen):
            counts.update(group_counts[key])
        score = sum((4 if k == 'size' else 1) * abs(counts[k]/v - fraction)
                    for k, v in totals.items())
        if score < best_score:
            best, best_score = chosen, score
    train = [r for r in rows if r['group_id'] not in best]
    val = [r for r in rows if r['group_id'] in best]
    return train, val, best_score


In [ ]:
def save_split(root, name, train, val, **meta):
    write_data_json(root / 'splits' / (name + '.json'), {
        **meta, 'train_ids': [r['id'] for r in train],
        'validation_ids': [r['id'] for r in val],
        'train_summary': summarize(train), 'validation_summary': summarize(val),
    })


### 2.6 数据处理流程


In [ ]:
def prepare_data(records, output_dir, seed=42, clean_description=False,
                 validation_fraction=0.2, split_search_attempts=2000):
    run = Path(output_dir).resolve()
    records, cleaning = clean_description_block(records, clean_description)
    by_language = {lang: [r for r in records if r['language'] == lang]
                   for lang in ['EN', 'IT', 'NL', 'FA']}
    train_all = [r for r in records if r['source_split'] == 'training']
    run.mkdir(parents=True, exist_ok=True)
    write_data_json(run / 'description_cleaning.json', cleaning)
    data_dir = run / 'processed'
    data_dir.mkdir(exist_ok=True)
    for lang, members in by_language.items():
        with (data_dir / f'{lang}.jsonl').open('w', encoding='utf-8', newline='\n') as f:
            for r in members:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
    write_data_json(run / 'label_schema.json', {
        'stereotype': STEREOTYPE, 'hate_speech': HATE_MAP, 'scope': SCOPE,
        'identity_order': IDENTITIES, 'scope_ignore_index': -100,
    })
    train_parts, val_parts = [], []
    for i, lang in enumerate(['EN', 'IT', 'NL']):
        train, val, score = grouped_holdout(by_language[lang], validation_fraction,
                                            seed+i, split_search_attempts)
        train_parts.extend(train)
        val_parts.extend(val)
        save_split(run, f'grouped_{lang}', train, val,
                                         protocol='within_language_grouped', seed=seed+i,
                                         requested_fraction=validation_fraction, balance_score=score)
    save_split(run, 'grouped_multilingual', train_parts, val_parts,
                                     protocol='within_language_grouped_combined', seed=seed)
    for held in ['EN', 'IT', 'NL']:
        train = [r for r in train_all if r['language'] != held]
        save_split(run, f'lolo_{held}', train, by_language[held],
                                         protocol='leave_one_language_out', held_out_language=held)
    write_data_json(run / 'splits' / 'persian_external.json', {
        'evaluation_ids': [r['id'] for r in by_language['FA']],
    })
    write_data_json(run / 'statistics.json', {lang: summarize(rows) if rows else {'n': 0} for lang, rows in by_language.items()})
    return run


## 3. 读取数据与分布统计


In [ ]:
records = load_data(DATA_DIR)
print("样本数量：", len(records))
for language in ["EN","IT","NL","FA"]:
    rows = [r for r in records if r["language"] == language]
    print(language, summarize(rows) if rows else {"n":0})

## 4. 推广内容统计与清洗预览


In [ ]:
audit = audit_content(records)
print({k:v for k,v in audit.items() if k != "duplicates"})
_, cleaning = clean_description_block(records, enabled=True)
print("清洗预览：", cleaning)

## 5. 长度与截断统计


### 5.1 统计配置


In [ ]:
USE_TOKENIZER = False
RUN_LENGTH_ANALYSIS = True
DEFAULT_CONFIG = {'input_mode': 'full', 'max_length': 512, 'comment_budget': 320, 'title_budget': 64, 'description_budget': 112}


### 5.2 长度统计与输入编码


In [ ]:
def percentile95(values):
    values = sorted(values)
    return values[math.ceil(0.95 * len(values)) - 1]


In [ ]:
def encode_record(row, tokenizer, config):
    fields = ['comment']
    if config['input_mode'] in ['comment_title', 'full']:
        fields.append('title')
    if config['input_mode'] == 'full':
        fields.append('description')
    limit = config['max_length'] - tokenizer.num_special_tokens_to_add(pair=False)
    marker_ids = {f: tokenizer.encode(f'[{f.upper()}] ', add_special_tokens=False) for f in fields}
    available = limit - sum(len(v) for v in marker_ids.values())
    if available < len(fields):
        raise ValueError('词元预算不足以容纳字段标记')
    raw = {f: tokenizer.encode(row[f], add_special_tokens=False) for f in fields}
    # 先给评论分配预算，再分配标题和描述；剩余容量
    # 优先返还评论，不直接截断拼接文本的尾部。
    if len(fields) == 1:
        budgets = {'comment': available}
    else:
        budgets = {'comment': min(config['comment_budget'], available-len(fields)+1)}
        remaining = available - min(len(raw['comment']), budgets['comment'])
        budgets['title'] = min(config['title_budget'], max(0, remaining-(1 if 'description' in fields else 0)))
        remaining -= min(len(raw['title']), budgets['title'])
        if 'description' in fields:
            budgets['description'] = min(config['description_budget'], remaining)
            remaining -= min(len(raw['description']), budgets['description'])
        if remaining > 0:
            extra = min(remaining, max(0, len(raw['comment'])-budgets['comment']))
            budgets['comment'] += extra
            remaining -= extra
            if 'description' in fields:
                budgets['description'] += remaining
    tokens, truncated = [], {}
    for field in fields:
        tokens += marker_ids[field] + raw[field][:budgets[field]]
        truncated[field] = len(raw[field]) > budgets[field]
    ids = tokenizer.build_inputs_with_special_tokens(tokens)
    if len(ids) > config['max_length']:
        raise ValueError('超出分词器长度预算')
    return {'input_ids': ids, 'attention_mask': [1]*len(ids)}, truncated


### 5.3 长度统计输出


In [ ]:
def analyze_lengths(records, use_tokenizer=False):
    tokenizer = None
    if use_tokenizer:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained('FacebookAI/xlm-roberta-base')

    groups = {language: [r for r in records if r['language'] == language]
              for language in ['EN', 'IT', 'NL', 'FA']}
    groups['TRAIN_ALL'] = [r for r in records if r['source_split'] == 'training']
    for language, rows in groups.items():
        lengths = {field: percentile95([len(r[field]) for r in rows])
                   for field in ['comment', 'title', 'description']}
        print(f"{language} | 样本 {len(rows)} | 字符 P95："
              f"评论 {lengths['comment']}，标题 {lengths['title']}，"
              f"描述 {lengths['description']}")
        if tokenizer:
            token_lengths, truncated = [], 0
            for row in rows:
                item, flags = encode_record(row, tokenizer, DEFAULT_CONFIG)
                token_lengths.append(len(item['input_ids']))
                truncated += any(flags.values())
            print(f"  编码长度 P95 {percentile95(token_lengths)}，最大 {max(token_lengths)} | "
                  f"截断 {truncated}/{len(rows)} ({100 * truncated / len(rows):.2f}%)")


### 5.4 执行统计


In [ ]:
if RUN_LENGTH_ANALYSIS:
    analyze_lengths(records, use_tokenizer=USE_TOKENIZER)

## 6. 处理并保存数据


In [ ]:
SEED = 42
CLEAN_DESCRIPTION = False
# 固定保存位置；重新运行会更新处理结果。
PREPARED_DIR = PROJECT /"experiments/prepared_data"
saved_dir = prepare_data(records, PREPARED_DIR, SEED, CLEAN_DESCRIPTION)
print("处理后的数据已保存到：", saved_dir)
